## Convergence check for four independent MP-MCMC chains

This notebook reads the saved `.npy` output files from the `outputdata/` directory and summarizes results from **four** independent Markov chains.

**What this does**
- Automatically loads chain outputs from `outputdata/`
- Computes standard convergence and mixing diagnostics across the four chains
- Produces plots and summary tables to help verify that the chains are sampling the same posterior distribution

**Why we run four chains**
Running multiple chains started from different random seeds helps detect:
- poor mixing (chains explore the space too slowly)
- multimodality (chains get stuck in different regions)
- lack of convergence (chains disagree on parameter values)

**Recommendation**
For reproducible results, run four chains with different `seed` values, and only interpret posterior summaries after convergence diagnostics indicate acceptable agreement across chains. We also recomend torun this notebook from the directory where `BayFlux` configuration files (`.yaml` format) are located. 

*Soumyadeep Ghosh, Tyler W. H. Backman*

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
# Restore the removed alias before importing cobra
np.object = object
np.bool = bool
np.int = int
np.float = float
np.complex = complex
np.str = str


import cobra
import cobra.core
import cobra.io
import math
from cobra.flux_analysis import flux_variability_analysis
import bayflux
import pandas as pd
import re
import os
from scipy.stats import norm
import operator
import seaborn as sns
import matplotlib.pyplot as plt
import yaml
import ntpath
from cobra.sampling import MCMCACHRSampler
from matplotlib.ticker import AutoLocator, AutoMinorLocator, FixedLocator
import time
from tabulate import tabulate
import arviz as az
import xarray as xr
import concurrent.futures
import multiprocessing as mp
from functools import partial
from concurrent.futures import ThreadPoolExecutor
from sklearn.decomposition import PCA
from IPython.display import display, clear_output

sns.set(style="white")
pd.set_option('display.max_rows', 1000)

In [ ]:
matplotlib.rcParams['figure.figsize'] = (16,8)

#### Create a directory named `analysisResults` to save the results

In [ ]:
analysisResultsDir = os.path.join(os.getcwd(), "analysisResults")
os.makedirs(analysisResultsDir, exist_ok=True)
print("Analysis results will be saved here:", analysisResultsDir)
individualDir = os.path.join(analysisResultsDir, "individualMarginalDistributions")
os.makedirs(individualDir, exist_ok=True)
print("Individual marginal distribution plots will be saved here:", individualDir)

#### Load configuration files 

- `configFiles` names need to be modified by user

In [ ]:
configFiles = ['config_job1.yaml', 'config_job2.yaml', 'config_job3.yaml', 'config_job4.yaml']

In [ ]:
# Loading data to a single file with different random seeds for analysis
configDataDict = {}  # Initialize an empty dictionary to store the configurations
for configFile in configFiles:
    with open(configFile, 'r') as stream:
        try:
            # Load the current configuration file
            configData = yaml.safe_load(stream)
            key = configFile.split('.')[0] 
            # Add the configuration to the dictionary
            configDataDict[key] = configData
        except yaml.YAMLError as exception:
            print(f"Error loading {configFile}: {exception}")

print(f"Number of unique job configurations: {len(configDataDict)}")


# Initialize an empty list to store all matching files across configurations
allMatchingFiles = []
# Initialize an empty list to store the extracted integers
fileNameIntegers = []

outputFolderChecked = False  # Flag to check if output folder has been listed

# Iterate over each configuration in configDataDict
for config in configDataDict.values():
    if not outputFolderChecked:
        outputFolder = ntpath.dirname(config['outputBaseName'])
        files = os.listdir(outputFolder)
        outputFolderChecked = True  # Set the flag after listing the directory

    baseNamePattern = re.escape(ntpath.basename(config['outputBaseName'])) + r'(\d{3}).npy$'
    
    # Iterate through files once, matching and extracting in one go
    for filename in files:
        match = re.match(baseNamePattern, filename)
        if match:
            allMatchingFiles.append(filename)
            # Directly extract and store the integer part of the filename
            fileNameIntegers.append(int(match.group(1)))

# Remove duplicates from lists if needed
allMatchingFiles = list(set(allMatchingFiles))

# Resume from the largest one only
if fileNameIntegers:
    max_value = max(fileNameIntegers)
    resumeHere = [max_value] * len(configDataDict)
else:
    print("No files to resume from. Starting fresh.")
    resumeHere = []

# Define function to generate an output filename from a number
def integerToFilename(outputBaseName, x, zfill=3):
    return outputBaseName + str(x).zfill(zfill) + '.npy'

# Initialize a list to hold the generated filenames
sampleFilenames = []

# Iterate over the items in configDataDict and the corresponding resumeHere value
for i, ((config_name, config), resume_value) in enumerate(zip(configDataDict.items(), resumeHere)):
    outputBaseName = config['outputBaseName']
    filename = integerToFilename(outputBaseName, resume_value)
    sampleFilenames.append(filename)

# Print the filenames
for index, filename in enumerate(sampleFilenames, start=1):
    print(f"Data file {index}: {filename}")

numProposals = int((configData['numProposals']))
print(f"\n Number of parallel proposals: {numProposals}")

#### Load the genome scale model from `cobrapy`

In [ ]:
cobrapymodel = cobra.io.read_sbml_model(configData['modelFile'])
model = bayflux.ReactionNetwork(cobrapymodel)
model

#### Choose how often to sample points from the saved output (subsampling interval)

This setting controls how many data points are selected from each chain for the analysis.

- larger interval (coarser subsampling) selects fewer data points → **faster** and uses **less memory**
- smaller interval (finer subsampling) selects more data points → **slower** but uses **high memory**

Recommendation: Start with a coarse interval to get quick diagnostics. If results look stable and you need more detailed resolution, decrease the interval and re-run the analysis.

In [ ]:
dataPointsToSelect = 1

In [ ]:
"""Load and preprocess MCMC chain outputs (.npy), then build per-chain DataFrames and net-flux arrays."""

startTime = time.time()

discardFraction = 0.25
print(f"burn-in initial {discardFraction*100}% samples")

def loadAndProcessFile(filename, discardFraction=discardFraction, dataPointsToSelect=10):
    """Load one chain, discard burn-in, then subsample (keep every Nth sample)."""
    try:
        if not os.path.isfile(filename):
            print(f"Warning: File not found or not accessible: {filename}")
            return None

        data = np.load(filename)
        discardCount = int(discardFraction * data.shape[0])
        processedData = data[discardCount::dataPointsToSelect]

        
        print(f"Shape of {filename}: after burn-in + subsampling -> {processedData.shape}")
        return processedData

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return None


if __name__ == "__main__":
    """Parallel-load all chain files listed in `sampleFilenames`."""
    numCores = max(1, int(mp.cpu_count() / 4))
    print(f"Using {numCores} CPU cores for parallel processing")

    with mp.Pool(processes=numCores) as pool:
        dataList = pool.map(
            partial(
                loadAndProcessFile,
                discardFraction=0.25,
                dataPointsToSelect=dataPointsToSelect,
            ),
            sampleFilenames,
        )

    dataList = [arr for arr in dataList if arr is not None]
    print(f"Successfully loaded {len(dataList)} chains" if dataList else "No data was successfully loaded!")

    print(f"Total processing time: {time.time() - startTime:.2f} seconds")
    print("Reading data completed ...")


# Per-chain processing: DataFrames + net flux (forward - reverse, assuming paired columns)
individualChainDataFrames = []
individualChainNetFluxes = []

for i, chainData in enumerate(dataList):
    """Convert chain samples to a DataFrame and compute net fluxes from paired columns."""
    chainDF = pd.DataFrame(chainData, columns=[v.name for v in model.variables])
    netChain = np.subtract(chainDF.iloc[:, 0::2], chainDF.iloc[:, 1::2])

    individualChainDataFrames.append(chainDF)
    individualChainNetFluxes.append(netChain)

    print(f"Chain {i+1}: samples {chainDF.shape}, net fluxes {netChain.shape}")


# Combined chain array (pooled samples after per-chain burn-in + subsampling)
combinedData = np.concatenate(dataList, axis=0)
numChains = len(dataList)

print(f"Combined data shape: {combinedData.shape}")
print(f"Found {numChains} Markov chains!")
print(f"Total time: {time.time() - startTime:.2f} s")

#### Now we convert this model to a bayflux ReactionNetwork, which inherets 'EnhancedReaction' objects from each cobra.Reaction, allowing us to add in atom transitions

In [ ]:
# Import and apply exchange fluxes from file
model.readFluxConstraints(configData['fluxBoundsFile'])

# set max flux
limit = configData['maxFlux']
for reaction in model.reactions:
    if reaction.lower_bound < -limit:
        reaction.lower_bound = -limit
    if reaction.upper_bound > limit:
        reaction.upper_bound = limit

# run FBA to confirm model works
model.optimize()

In [ ]:
mcmcachr = cobra.sampling.MCMCACHRSampler(model)

#### Generate uniform samples for comparison

This step generates a set of uniformly sampled representative points to use as a baseline when comparing against the MCMC samples (for example, to check whether the chains explore the feasible space).

- This can be slow for large genome-scale models because it may require generating and storing many samples.
- For repeat analyses, we recommend reusing previously generated uniform samples saved as `.pkl` files instead of regenerating them each time. Just load it here and skip the generation step

In [ ]:
warmupSamplesFile = "warmupSamples.pkl"
warmupSamplesToCollect = 100000

# Check if warmup samples exist, otherwise generate them
if os.path.exists(warmupSamplesFile):
    try:
        warmupSamples = pd.read_pickle(warmupSamplesFile)
        
        if warmupSamples.shape[0] == warmupSamplesToCollect:
            print(f"Loaded warmupSamples from {warmupSamplesFile} file with shape {warmupSamples.shape}.")
        else:
            raise ValueError("File exists but does not contain the correct number of samples.")
    except Exception as e:
        print(f"Error loading file: {e}")
        print("Recomputing warmupSamples...")
        warmupSamples = mcmcachr.sample(warmupSamplesToCollect, fluxes=False, likelihood=False)
        
        # Save DataFrame with pickle (preserves all metadata including column names)
        warmupSamples.to_pickle(warmupSamplesFile)
        print("Saved newly computed warmupSamples to file.")
else:
    print("File not found. Computing warmupSamples...")
    warmupSamples = mcmcachr.sample(warmupSamplesToCollect, fluxes=False, likelihood=False)
    
    # Save DataFrame with pickle (preserves all metadata including column names)
    warmupSamples.to_pickle(warmupSamplesFile)
    print("Saved warmupSamples to file.")

print("WarmupSamples shape:", warmupSamples.shape)

In [ ]:
warmupSamples

#### Load the samples in a Pandas DF

In [ ]:
realSamples = pd.DataFrame(combinedData, columns=[v.name for v in model.variables])
del combinedData

In [ ]:
realSamples

#### To plot marginals and trace we will just plot the last 90% of the first markov chain

In [ ]:
print(f"Shape of combined data from {numChains} markov chains: {realSamples.shape}")

In [ ]:
netFirstChain = np.subtract(realSamples.iloc[:,0::2], realSamples.iloc[:,1::2])
print(f"Total net samples (flux for forward reactions - flux for backward reactions): {netFirstChain.shape}")

#### Create a list of the reactions to plot fluxes
- choose some specific reactions
- choose all reactions

In [ ]:
allRxns = ['PGI','EDD','AKGDH', 'G6PDH2r']

# allRxns = [r.id for r in model.reactions][0:4]

In [ ]:
def saveFigure(fig, analysisResultsDir, baseName, dpi=1200):
    """Save a Matplotlib figure as PNG and SVG in analysisResultsDir."""
    os.makedirs(analysisResultsDir, exist_ok=True)
    outPng = os.path.join(analysisResultsDir, f"{baseName}.png")
    outSvg = os.path.join(analysisResultsDir, f"{baseName}.svg")
    fig.savefig(outPng, dpi=dpi, bbox_inches="tight")
    fig.savefig(outSvg, bbox_inches="tight")
    print(f"Saved: {outPng} and {outSvg}")


# Individual chain
def plotIndividualChainTraces(chainNetFluxDFList, reactionList, analysisResultsDir, maxPlotSamples=1000):
    """Plot per-chain trace plots (one subplot per reaction) and save the figure."""
    numChains = len(chainNetFluxDFList)
    numReactions = len(reactionList)
    chainColors = plt.cm.tab10(np.linspace(0, 1, numChains))

    fig, axes = plt.subplots(numReactions, 1, figsize=(12, 3 * numReactions))
    if numReactions == 1:
        axes = [axes]

    for rxnIdx, reactionName in enumerate(reactionList):
        ax = axes[rxnIdx]

        if reactionName not in chainNetFluxDFList[0].columns:
            ax.text(0.5, 0.5, f"Reaction {reactionName} not found",
                    transform=ax.transAxes, ha="center", va="center")
            ax.set_title(f"Individual Chain Traces - {reactionName} (Not Found)")
            ax.axis("off")
            continue

        for chainIdx, netFluxDF in enumerate(chainNetFluxDFList):
            numSamplesToPlot = min(len(netFluxDF), maxPlotSamples)
            sampleIdx = np.linspace(0, len(netFluxDF) - 1, numSamplesToPlot, dtype=int)

            ax.plot(
                sampleIdx,
                netFluxDF[reactionName].iloc[sampleIdx],
                color=chainColors[chainIdx],
                alpha=0.8,
                linewidth=1,
                label=f"Chain {chainIdx+1}",
            )

        ax.set_title(f"Individual Chain Traces - {reactionName}")
        ax.set_xlabel("Samples")
        ax.set_ylabel("Net Flux")
        ax.legend(frameon=True)
        ax.grid(True, alpha=0.3)

    fig.tight_layout()
    saveFigure(fig, analysisResultsDir, baseName="trace_individualChains_selectedRxns")
    plt.show()
    plt.close(fig)


def plotChainComparisonStatistics(chainNetFluxDFList, reactionList, analysisResultsDir):
    """Compare chain means/std + distribution + running mean (first valid reaction) and save the figure."""
    numChains = len(chainNetFluxDFList)
    chainColors = plt.cm.tab10(np.linspace(0, 1, numChains))

    validRxnList = [r for r in reactionList if r in chainNetFluxDFList[0].columns]
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    if not validRxnList:
        axes[0, 0].text(0.5, 0.5, "No requested reactions found in data",
                        transform=axes[0, 0].transAxes, ha="center", va="center")
        for ax in axes.ravel():
            ax.axis("off")
        fig.tight_layout()
        saveFigure(fig, analysisResultsDir, baseName="chainComparison_statistics")
        plt.show()
        plt.close(fig)
        return

    # --- Mean comparison ---
    meanMatrix = np.array([[df[rxn].mean() for df in chainNetFluxDFList] for rxn in validRxnList])
    xPos = np.arange(len(validRxnList))
    barWidth = 0.8 / numChains

    for chainIdx in range(numChains):
        axes[0, 0].bar(
            xPos + chainIdx * barWidth,
            meanMatrix[:, chainIdx],
            barWidth,
            label=f"Chain {chainIdx+1}",
            color=chainColors[chainIdx],
            alpha=0.7,
        )
    axes[0, 0].set_title("Mean Net Flux Comparison Across Chains")
    axes[0, 0].set_xlabel("Reactions")
    axes[0, 0].set_ylabel("Mean Net Flux")
    axes[0, 0].set_xticks(xPos + barWidth * (numChains - 1) / 2)
    axes[0, 0].set_xticklabels(validRxnList, rotation=45, ha="right")
    axes[0, 0].legend(frameon=True)
    axes[0, 0].grid(True, alpha=0.3)

    # --- Std comparison ---
    stdMatrix = np.array([[df[rxn].std() for df in chainNetFluxDFList] for rxn in validRxnList])
    for chainIdx in range(numChains):
        axes[0, 1].bar(
            xPos + chainIdx * barWidth,
            stdMatrix[:, chainIdx],
            barWidth,
            label=f"Chain {chainIdx+1}",
            color=chainColors[chainIdx],
            alpha=0.7,
        )
    axes[0, 1].set_title("Standard Deviation Comparison Across Chains")
    axes[0, 1].set_xlabel("Reactions")
    axes[0, 1].set_ylabel("Standard Deviation")
    axes[0, 1].set_xticks(xPos + barWidth * (numChains - 1) / 2)
    axes[0, 1].set_xticklabels(validRxnList, rotation=45, ha="right")
    axes[0, 1].legend(frameon=True)
    axes[0, 1].grid(True, alpha=0.3)

    firstRxn = validRxnList[0]

    # --- Distribution (violin) for first reaction ---
    violinData = [df[firstRxn].values for df in chainNetFluxDFList]
    violinParts = axes[1, 0].violinplot(violinData, positions=range(1, numChains + 1))
    for i, body in enumerate(violinParts["bodies"]):
        body.set_facecolor(chainColors[i])
        body.set_alpha(0.7)
    axes[1, 0].set_title(f"Distribution Comparison - {firstRxn}")
    axes[1, 0].set_xlabel("Chain")
    axes[1, 0].set_ylabel("Net Flux")
    axes[1, 0].set_xticks(range(1, numChains + 1))
    axes[1, 0].set_xticklabels([f"Chain {i+1}" for i in range(numChains)])
    axes[1, 0].grid(True, alpha=0.3)

    # --- Running mean for first reaction ---
    windowSize = 100
    for chainIdx, df in enumerate(chainNetFluxDFList):
        runningMean = df[firstRxn].rolling(window=windowSize).mean()
        axes[1, 1].plot(runningMean, color=chainColors[chainIdx], label=f"Chain {chainIdx+1}", alpha=0.8)
    axes[1, 1].set_title(f"Running Mean - {firstRxn} (window={windowSize})")
    axes[1, 1].set_xlabel("Samples")
    axes[1, 1].set_ylabel("Running Mean")
    axes[1, 1].legend(frameon=True)
    axes[1, 1].grid(True, alpha=0.3)

    fig.tight_layout()
    saveFigure(fig, analysisResultsDir, baseName="chainComparison_statistics")
    plt.show()
    plt.close(fig)


def plotEachChainAllReactions(chainNetFluxDFList, reactionList, analysisResultsDir, maxPlotSamples=1000):
    """Plot all requested reactions for each chain (one figure per chain) and save each figure."""
    for chainIdx, netFluxDF in enumerate(chainNetFluxDFList):
        fig, ax = plt.subplots(figsize=(15, 8))

        allYValues = []
        for reactionName in reactionList:
            if reactionName not in netFluxDF.columns:
                continue

            numSamplesToPlot = min(len(netFluxDF), maxPlotSamples)
            sampleIdx = np.linspace(0, len(netFluxDF) - 1, numSamplesToPlot, dtype=int)

            yValues = netFluxDF[reactionName].iloc[sampleIdx]
            allYValues.extend(yValues)
            ax.plot(sampleIdx, yValues, linewidth=1.2, alpha=1, label=reactionName)

        ax.set_title(f"Chain {chainIdx+1} - All Reactions Trace Plot", fontsize=16, fontweight="bold")
        ax.set_xlabel("Samples", fontsize=12)
        ax.set_ylabel("Net Flux", fontsize=12)
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
        ax.grid(True, alpha=0.3)

        if allYValues:
            yMin, yMax = min(allYValues), max(allYValues)
            yRange = yMax - yMin
            ax.set_ylim(yMin - 0.1 * yRange, yMax + 0.1 * yRange)

        fig.tight_layout()
        saveFigure(fig, analysisResultsDir, baseName=f"trace_chain{chainIdx+1}_allRxns")
        plt.show()
        plt.close(fig)


# Combined chain
def plotCombinedChainTraces(netFluxDF, reactionList, analysisResultsDir, maxPlotSamples=2000):
    """Plot combined-chain traces for requested reactions and save the figure."""
    fig, ax = plt.subplots(figsize=(15, 8))

    allYValues = []
    for reactionName in reactionList:
        if reactionName not in netFluxDF.columns:
            continue

        numSamplesToPlot = min(len(netFluxDF), maxPlotSamples)
        sampleIdx = np.linspace(0, len(netFluxDF) - 1, numSamplesToPlot, dtype=int)

        yValues = netFluxDF[reactionName].iloc[sampleIdx]
        allYValues.extend(yValues)
        ax.plot(sampleIdx, yValues, linewidth=1.2, alpha=1, label=reactionName)

    ax.set_title("Combined Chain Traces - All Reactions", fontsize=16, fontweight="bold")
    ax.set_xlabel("Samples", fontsize=12)
    ax.set_ylabel("Net Flux", fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
    ax.grid(True, alpha=0.3)

    if allYValues:
        yMin, yMax = min(allYValues), max(allYValues)
        yRange = yMax - yMin
        ax.set_ylim(yMin - 0.1 * yRange, yMax + 0.1 * yRange)

    fig.tight_layout()
    saveFigure(fig, analysisResultsDir, baseName="trace_combinedChains_allRxns")
    plt.show()
    plt.close(fig)


# Run
print("Individual chain analysis")
print("-" * 30)

plotIndividualChainTraces(individualChainNetFluxes, allRxns, analysisResultsDir)
plotChainComparisonStatistics(individualChainNetFluxes, allRxns, analysisResultsDir)
plotEachChainAllReactions(individualChainNetFluxes, allRxns, analysisResultsDir)

print("Combined analysis of all chains")
print("-" * 30)

plotCombinedChainTraces(netFirstChain, allRxns, analysisResultsDir)

#### PairGrid plot of MP-MCMC samples: lower triangle hexbin + diagonal KDE

In [ ]:
g = sns.PairGrid(netFirstChain[allRxns], diag_sharey=False)
g.map_lower(plt.hexbin, gridsize=25, cmap=plt.get_cmap("Blues"), linewidths=0)
g.map_diag(sns.kdeplot, lw=3, gridsize=100)

# Hide upper triangle
for i, j in zip(*np.triu_indices_from(g.axes, 1)):
    g.axes[i, j].set_visible(False)

g.fig.tight_layout()
display(g.fig)

outPng = os.path.join(analysisResultsDir, "pairgrid_hexbin_kde.png")
outSvg = os.path.join(analysisResultsDir, "pairgrid_hexbin_kde.svg")

g.fig.savefig(outPng, dpi=1200, bbox_inches="tight")
g.fig.savefig(outSvg, bbox_inches="tight")

print(f"Saved PairGrid plot to:\n  {outPng}\n  {outSvg}")
plt.close(g.fig)

# Analyze sampler convergence with Gelman–Rubin diagnostic

Each chain sampling from the posterior effectively starts at a different random location from a uniform sample of the flux space because of the warmupSamples, which end at a different spot depending on the random seed. Values close to 1.0 (for example 1.1 or less) typically indicate convergence.

In [ ]:
deleteStart = 0
netSamples = np.subtract(realSamples.iloc[deleteStart::1,0::2], realSamples.iloc[deleteStart::1,1::2])
netUniform = np.subtract(warmupSamples.iloc[deleteStart::1,0::2], warmupSamples.iloc[deleteStart::1,1::2])
print(f"Total net MP-MCMC samples (flux for forward reactions - flux for backward reactions): {netSamples.shape}")
print(f"Total net uniform samples (flux for forward reactions - flux for backward reactions): {netUniform.shape}")

#### Remove reactions with no variance, e.g. they cannot carry flux or were fully constrained

In [ ]:
rxnsWithVariance = netUniform.std(axis=0) > 0.001
netUniform = netUniform.loc[:,rxnsWithVariance]
netSamples = netSamples.loc[:,rxnsWithVariance]
print(f"Total net MP-MCMC samples with variance: {netSamples.shape}")
print(f"Total net uniform samples with variance: {netUniform.shape}")

#### Compute thinning intervals for MP-MCMC and uniform samples (currently defaults to 1)

In [ ]:
warmupSampleThinning = math.ceil(warmupSamples.shape[0]/warmupSamples.shape[0])
print(f"warmupSampleThinning: {warmupSampleThinning}")
realSampleThinning = math.ceil(netSamples.shape[0]/netSamples.shape[0])
print(f"realSampleThinning: {realSampleThinning}")

In [ ]:
# compute stat with net fluxes only
# comment these out if we want to analyze each direction independently
numpysamples = np.array(realSamples)
data = np.subtract(numpysamples[:,0::2], numpysamples[:,1::2])
data.shape

In [ ]:
samplesPerChain = int(data.shape[0]/numChains)
print(f"samples per chain: {samplesPerChain}")

#### Compute and plot Gelman–Rubin diagnostic

In [ ]:
%%time

# Define the number of samples and thinning interval
thin = int(math.ceil(samplesPerChain / 10))

# Reshape data into a 3D array where each 1st dimension is a chain as used by pymc
samples = data.shape[0]
data3D = np.reshape(data, newshape=(numChains, samplesPerChain, -1), order='C')

# Initialize an array to store the Gelman-Rubin statistics
grStats = np.zeros(shape=(int(samplesPerChain / thin), data3D.shape[2]), dtype=np.float64, order='C')

# Iteratively analyze convergence with stepsize thin using arviz's rhat function
for point in range(1, int(samplesPerChain / thin) + 1):
    thisPoint = int(point * thin)
    subset = np.take(data3D, range(thisPoint), axis=1)
    print(subset.shape)

    # Convert the subset into a DataArray and then an InferenceData object
    subset_xr = xr.DataArray(subset, dims=["chain", "draw", "variable"])
    idata = az.convert_to_inference_data(subset_xr)

    # Compute Gelman-Rubin diagnostic (R-hat) using arviz
    grStats[point - 1] = az.rhat(idata).to_array().values

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

pd.DataFrame(grStats).plot(ax=ax, legend=False)

ax.set_xlabel("Iteration")
ax.set_ylabel("GR statistics ($\hat{R}$)")
ax.set_title("Gelman–Rubin (GR) Diagnostic Over Simulation")
ax.set_ylim(1)  
fig.tight_layout()
plt.show()


outPng = os.path.join(analysisResultsDir, "gr_diagnostic_over_time.png")
outSvg = os.path.join(analysisResultsDir, "gr_diagnostic_over_time.svg")
fig.savefig(outPng, dpi=1200, bbox_inches="tight")
fig.savefig(outSvg, bbox_inches="tight")

print(f"Saved GR diagnostic plot to:\n  {outPng}\n  {outSvg}")
plt.close(fig)

In [ ]:
final_gr = grStats[-1, :]  # last iteration

fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(final_gr, bins=30, edgecolor='black', color='blue', alpha=0.7)
ax.axvline(x=1.1, linestyle='--', color='red', lw=1.5, label='$\hat{R}$ = 1.1 threshold')
#ax.axvline(x=np.median(final_gr), linestyle='--', color='darkgreen', lw=1.5, 
#           label=f'Median = {np.median(final_gr):.4f}')

ax.set_xlabel('Final $\hat{R}$')
ax.set_ylabel('Number of reactions')
ax.legend(frameon=False, fontsize='small')
ax.minorticks_on()
ax.tick_params(which='both', direction='out', bottom=True, left=True, top=False, right=False)
ax.tick_params(which='minor', length=4, color='k')
ax.tick_params(which='major', length=7, color='k')

pct_converged = 100 * np.sum(final_gr < 1.1) / len(final_gr)
ax.text(0.95, 0.95, f'{pct_converged:.1f}% < 1.1', transform=ax.transAxes,
        ha='right', va='top', fontsize=11, fontweight='bold')

plt.tight_layout()
fig.savefig("gr_final_histogram.png", dpi=1200, bbox_inches='tight')
fig.savefig("gr_final_histogram.pdf", dpi=1200, bbox_inches='tight')
plt.show()

#### Report GR-statistics for each reaction to summarize convergence across chains

In [ ]:
GRbyVariable = pd.DataFrame(grStats[grStats.shape[0] - 1], index=[v.name for v in model.variables][0::2], columns=['GR']).dropna()
maskAligned = rxnsWithVariance.reindex(GRbyVariable.index, fill_value=False)
rxnsWithVarianceDF = GRbyVariable.loc[maskAligned].sort_values("GR")
rxnsWithVarianceDF

#### Summary of Gelman–Rubin (GR) convergence across reactions

In [ ]:
numRxns = GRbyVariable.shape[0]
print(f"Total reactions evaluated: {numRxns}")

numConverged = (GRbyVariable["GR"] < 1.10).sum()
numNotConverged = (GRbyVariable["GR"] > 1.10).sum()

print(f"Converged reactions (GR < 1.10): {numConverged} ({numConverged/numRxns*100:.2f}%)")
print(f"Non-converged reactions (GR > 1.10): {numNotConverged} ({numNotConverged/numRxns*100:.2f}%)")

# Restrict to reactions that show variance (align the mask to avoid index mismatch)
maskAligned = rxnsWithVariance.reindex(GRbyVariable.index, fill_value=False)
GRwithVariance = GRbyVariable.loc[maskAligned, "GR"]

numVarRxns = GRwithVariance.shape[0]
numVarConverged = (GRwithVariance < 1.10).sum()
numVarNotConverged = (GRwithVariance > 1.10).sum()

print(f"\nNo. of reactions with variance: {numVarRxns}")
print(f"Converged (with variance, GR < 1.10): {numVarConverged} ({numVarConverged/numVarRxns*100:.2f}%)")
print(f"Non-converged (with variance, GR > 1.10): {numVarNotConverged} ({numVarNotConverged/numVarRxns*100:.2f}%)")

#### Save converged reaction names to a text file (one per line)

In [ ]:
convergedRxnsMpmcmcEMU = (rxnsWithVarianceDF.loc[rxnsWithVarianceDF["GR"] < 1.10].sort_values("GR").index.tolist())

print(f"Converged reactions (with variance, GR < 1.10): {len(convergedRxnsMpmcmcEMU)}")

outputFilename = os.path.join(analysisResultsDir, "convergedRxns_mpmcmc_EMU.txt")
with open(outputFilename, "w") as file:
    file.write("\n".join(convergedRxnsMpmcmcEMU) + "\n")

print(f"Saved converged reaction list to: {outputFilename}")

#### Convergence over sampling progress

Compute R-hat (Gelman–Rubin) at 10–100% of the retained samples to track how many reactions meet the convergence threshold (R-hat < 1.10) as sampling increases.

In [ ]:
%%time

# Thinning interval
thinInterval = int(math.ceil(samplesPerChain / 10))

# Reshape pooled samples into (chain, draw, variable) for ArviZ/PyMC-style diagnostics
totalSamples = data.shape[0]
chainDrawVarData = np.reshape(data, newshape=(numChains, samplesPerChain, -1), order="C")

# Store convergence summaries at increasing fractions of the retained samples
convergenceSummaryList = []

# Evaluate convergence at 10% interval of retained samples per chain
#sampleFractions = np.arange(10, 110, 10)
sampleFractions = np.arange(5, 105, 5)
print("Burn-in note: the first 25% of samples from each chain were excluded before this analysis.")

for fractionPercent in sampleFractions:
    drawsToUse = int((fractionPercent / 100) * samplesPerChain)

    # Use the first `drawsToUse` draws from each chain
    subsetChainDrawVar = np.take(chainDrawVarData, range(drawsToUse), axis=1)

    # Convert to InferenceData for ArviZ diagnostics
    subsetDataArray = xr.DataArray(subsetChainDrawVar, dims=["chain", "draw", "variable"])
    inferenceData = az.convert_to_inference_data(subsetDataArray)

    # R-hat per variable
    rhatArray = az.rhat(inferenceData).to_array().values

    # Count converged variables (R-hat < 1.10)
    convergedCount = int(np.sum(rhatArray < 1.10))
    totalVariableCount = int(rhatArray.size)
    convergedPercent = round((convergedCount / totalVariableCount) * 100, 2)

    # Report an approximate "working sample count" (adjusted for burn-in removal)
    effectiveWorkingSamples = int(round((drawsToUse * numProposals * dataPointsToSelect) / 0.75))

    convergenceSummaryList.append(
        (effectiveWorkingSamples, fractionPercent, convergedCount, convergedPercent)
    )

    print(
        f"After {effectiveWorkingSamples} samples ({fractionPercent}% of retained draws): "
        f"{convergedCount}/{totalVariableCount} ({convergedPercent:.2f}%) reactions converged (R-hat < 1.10)"
    )

convergenceDF = pd.DataFrame(convergenceSummaryList,
    columns=[
        "EffectiveSamples",
        "PercentOfRetainedDraws",
        "ConvergedReactions",
        "%ConvergedReactions",
    ],
)

convergenceDF.to_csv(f"/bayflux/Test-6/3-ecoli_genome_scale-wt5h/MP-MCMC/Data4Paper/Plots/convergenceDF_N{numProposals}.csv")
convergenceDF

### Plot marginals for all converged reactions (with variance)

- Here we plot a marginal distribution for each reaction as a net flux, while excluding the first 25% of the samples, and by adding the forward and reverse fluxes to get net flux for reversible reactions
- Here we are considering only converged reactions from GR-statistics (GR < 1.10) and have variance
- Comparing uniform samples against MP-MCMC samples

In [ ]:
# Reactions to plot = converged reaction names (present as the index of rxnsWithVarianceDF)
#rxnsToPlot = convergedRxnsMpmcmcEMU

# Keep only reactions that exist in both datasets (uniform + MCMC net fluxes)
#rxnsToPlot = [r for r in rxnsToPlot if (r in netUniform.columns) and (r in netSamples.columns)]

# If you only want to plot top 20 lowest-GR reactions
#rxnsToPlot = rxnsWithVarianceDF.loc[rxnsWithVarianceDF["GR"] < 1.10].sort_values("GR").head(20).index.tolist()

# Plot specific reactions by 'reaction_id'
rxnsToPlot = ['GND', 'TALA', 'PGI', 'RPE', 'TPI', 'G6PDH2r','RPI','PGK','GAPD','TKT1','TKT2',
               'MTHFC', 'ACOTA', 'THRS', 'MCITS']

print(f"Reactions that will be plotted: {len(rxnsToPlot)}")

In [ ]:
# Validate reactions you want to plot
rxnsPresent = [r for r in rxnsToPlot if (r in netUniform.columns and r in netSamples.columns)]
missingRxns = [r for r in rxnsToPlot if r not in rxnsPresent]
if missingRxns:
    print("Skipping missing reactions:", missingRxns)

# Pre-slice data by row thinning and by reaction NAMES
uniform_data = netUniform.loc[0::warmupSampleThinning, rxnsPresent]
mcmc_data    = netSamples.loc[0::realSampleThinning,  rxnsPresent]

cols = 3
rows = math.ceil(len(rxnsPresent) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(16, 3 * rows), squeeze=False)

for j in range(rows):
    for k in range(cols):
        plotIdx = (j * cols) + k
        ax = axes[j, k]

        if plotIdx >= len(rxnsPresent):
            ax.axis("off")
            continue

        rxnName = rxnsPresent[plotIdx]

        sns.kdeplot(
            uniform_data[rxnName],
            ls="-", lw=2, gridsize=100, color="orange",
            label="uniform" if k == 0 else "",
            ax=ax,
        )

        sns.kdeplot(
            mcmc_data[rxnName],
            ls="-", lw=2, bw_adjust=1, gridsize=100, color="darkred",
            label="MP-MCMC" if k == 0 else "",
            ax=ax,
        )

        ax.set_xlabel(f"{rxnName} Flux")
        if k == 0:
            ax.set_ylabel("Probability density")
            ax.legend()

        ax.minorticks_on()
        ax.tick_params(which="both", direction="out", bottom=True, left=True, top=False, right=False)
        ax.tick_params(which="minor", length=4, color="k")
        ax.tick_params(which="major", length=7, color="k")

fig.tight_layout()
display(fig)

combinedPng = os.path.join(analysisResultsDir, "marginal_distributions_all.png")
combinedSvg = os.path.join(analysisResultsDir, "marginal_distributions_all.svg")
fig.savefig(combinedPng, dpi=1200, bbox_inches="tight")
fig.savefig(combinedSvg, bbox_inches="tight")
plt.close(fig)

print(f"Saved combined marginal distributions to:\n  {combinedPng}\n  {combinedSvg}")

### Individual marginal plots: one reaction per file

In [ ]:
def modifyRxnName(name: str) -> str:
    """Make reaction names safe for filenames."""
    return name.replace("/", "_").replace("\\", "_").replace(" ", "_")

rxnsPresent = list(uniform_data.columns)  # safest: use what is actually in the sliced DF

for plotNum, rxnName in enumerate(rxnsPresent, start=1):
    safeName = modifyRxnName(rxnName)

    fig, ax = plt.subplots(figsize=(6, 4))

    sns.kdeplot(
        uniform_data[rxnName],
        ls="-", lw=2, gridsize=100, color="orange",
        label="uniform",
        ax=ax,
    )

    sns.kdeplot(
        mcmc_data[rxnName],
        ls="-", lw=2, bw_adjust=1, gridsize=100, color="darkred",
        label="MP-MCMC",
        ax=ax,
    )

    ax.set_xlabel(f"{rxnName} Flux")
    ax.set_ylabel("Probability density")
    ax.legend(frameon=True)

    ax.minorticks_on()
    ax.tick_params(which="both", direction="out", bottom=True, left=True, top=False, right=False)
    ax.tick_params(which="minor", length=4, color="k")
    ax.tick_params(which="major", length=7, color="k")

    fig.tight_layout()

    outPng = os.path.join(individualDir, f"{safeName}_marginal.png")
    outSvg = os.path.join(individualDir, f"{safeName}_marginal.svg")
    fig.savefig(outPng, dpi=1200, bbox_inches="tight")
    fig.savefig(outSvg, bbox_inches="tight")

    clear_output(wait=True)
    display(fig)
    print(f"[{plotNum}/{len(rxnsPresent)}] Saved: {outPng} and {outSvg}")

    plt.close(fig)

print(f"Saved individual marginal distributions in:\n  {individualDir}")